In [1]:
'''
notebook for salinity normalization of variables from GLODAPv2.2023_Merged_Master_File.csv
'''

'\nnotebook for salinity normalization of variables from GLODAPv2.2023_Merged_Master_File.csv\n'

In [ ]:
import warnings
warnings.filterwarnings('ignore') # don't output warnings
from pathlib import Path
import numpy as np
from pathlib import Path
import pandas as pd 

========================================================================================================================================================

User configs:

In [ ]:
glodap_dir = '/space/hall7/sitestore/eccc/crd/cccma/users/rpg002/data/GLODAP/GLODAPv2.2023_Merged_Master_File.csv'
output_dir = f'/space/hall7/sitestore/eccc/crd/cccma/users/rpg002/data'

var_list = ['talk', 'tco2']



========================================================================================================================================================

In [ ]:
data = pd.read_csv(glodap_dir)

In [ ]:
obs_to_model_rename = {'salinity' : 'so' , 'theta' : 'thetao', 'oxygen' :'o2', 'tco2' : 'dissic', 'nitrate' : 'no3', 'phosphate' : 'po4'}

def csv_var(data, var):
    
    columns_to_extract = ['G2year', 'G2month', 'G2day', 'G2hour',  'G2minute', 'G2latitude', 'G2longitude', 'G2depth', 'G2pressure', f'G2{var}']  
    var_data = data[columns_to_extract]
    rename_dict = {}
    for item in columns_to_extract:
        rename_dict[item] =  item.split('G2')[1]

    var_data.rename(columns=rename_dict, inplace=True)
    
    var_data[var] = var_data[var].replace(-9999, np.nan)
    var_data_columns = list(var_data.columns)
    var_data['date'] = pd.to_datetime(var_data[['year', 'month', 'day', 'hour', 'minute']])
    var_data = var_data[['date', *var_data_columns]]
    var_data.rename(columns={'month' : 'time', 'latitude' : 'lat', 'longitude' : 'lon', 'depth' : 'lev'}, inplace=True)
    var_data.rename(columns=obs_to_model_rename, inplace=True)
    return var_data



In [ ]:

var_data_so = csv_var(data, 'salinity')

for var in var_list:
    var_data = csv_var(data, var)


    var_data_normlaized = var_data.copy()
    var_data_normlaized[var] = var_data[var] * (35/var_data_so['so'])


    dirout = f'{output_dir}/n{var}/observation'
    Path(dirout).mkdir(parents=True, exist_ok=True)
    var_data_normlaized.to_csv(dirout + f'/GLODAP/GLODAP_v2_2023_n{var}_1984-2021.csv')
